# Análisis Nivel 3 — Representación ESCO de Carreras Universitarias

**Proyecto:** TIC-D — Análisis de similitud curricular entre carreras de la EPN y universidades OCDE  
**Nivel:** 3 de 5 — Representación basada en habilidades ESCO  
**Autor:** Glenn Salcedo 
**Institución:** Escuela Politécnica Nacional (EPN)

---

## ¿Qué hace este notebook?

Este notebook analiza la similitud curricular entre carreras universitarias usando **vectores de habilidades ESCO** como representación. 

A diferencia del Nivel 1 (TF-IDF), que compara palabras exactas, este nivel compara **qué competencias y habilidades** cubre cada carrera, usando la taxonomía europea ESCO (European Skills, Competences, Qualifications and Occupations) como vocabulario común.

### Pipeline del Nivel 3

```
Texto de carrera (perfil_egreso + perfil_profesional)
        ↓
Embedding semántico (BGE-M3 o E5-Large)
        ↓
Comparación contra 4.241 habilidades ESCO
        ↓
Top-25 habilidades más cercanas por carrera
        ↓
Vector de habilidades por carrera (163 × 4.241)
        ↓
Similitud carrera vs carrera (163 × 163)
        ↓
Heatmap + Dendrograma + Métricas
```

### Modelos evaluados

| Modelo | Descripción | Dimensión |
|--------|-------------|----------|
| **BGE-M3** (BAAI/bge-m3) | Modelo multilingüe de última generación, soporta hasta 8.192 tokens | 1.024 |
| **E5-Large** (intfloat/multilingual-e5-large) | Modelo multilingüe optimizado para similitud semántica | 1.024 |

### Estructura del notebook

Cada sección es **independiente** — puedes ejecutar solo la que necesitas sin correr todo el notebook. El único prerequisito es haber ejecutado la **Sección 0** (configuración) y la **Sección 1** (carga de datos).

---
## Sección 0 — Configuración e Imports

**Qué hace:** Importa todas las librerías necesarias y define las rutas a los archivos de datos y resultados.

**Cuándo ejecutar:** Siempre primero, antes de cualquier otra sección.

**Librerías utilizadas:**
- `pandas` / `numpy` — manipulación de datos y matrices
- `matplotlib` — generación de gráficos
- `scipy` — clustering jerárquico y métricas de distancia
- `sklearn` — similitud coseno y Silhouette score

In [ ]:
import glob
import json
import warnings
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet, fcluster, leaves_list
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────────────────────
BASE_DIR   = Path('.')                          # raíz del repositorio TIC-D
PROCESSED  = BASE_DIR / 'data' / 'processed'   # datos procesados
OUTPUT_DIR = BASE_DIR / 'outputs' / date.today().isoformat()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Paleta de colores por universidad ──────────────────────────────────────────
PALETA = {
    'EPN':'#1a56db','ESPOL':'#0e9f6e','UPS':'#e3a008','ESPOCH':'#9061f9',
    'UCE':'#e02424','UTM':'#ff5a1f','UG':'#057a55','USFQ':'#0694a2',
    'UDLA':'#c81e1e','UTPL':'#5521b5','ESPE':'#1e429f','UCSG':'#723b13',
    'PUCE':'#014737','UTN':'#6b21a8',
}
def get_color(sig): return PALETA.get(sig, '#4b5563')

print('✓ Configuración completada')
print(f'  Directorio base:    {BASE_DIR.resolve()}')
print(f'  Datos procesados:   {PROCESSED.resolve()}')
print(f'  Salida de hoy:      {OUTPUT_DIR.resolve()}')

---
## Sección 1 — Carga de Datos

**Qué hace:** Carga tres archivos generados previamente por `esco_mapping.py`:

1. **`carreras_homologas.csv`** — el dataset maestro con las 163 carreras y sus textos de perfil
2. **`esco_top25_MODELO_FECHA.csv`** — las 25 habilidades ESCO más cercanas a cada carrera
3. **`esco_vectors_MODELO_FECHA.npy`** — la matriz de similitud carrera × habilidades ESCO (163 × 4.241)

**Cuándo ejecutar:** Siempre antes de las secciones 2-7. Si cambias de modelo (BGE-M3 ↔ E5), vuelve a ejecutar esta sección.

**Parámetro configurable:** `MODELO` — cambia entre `'bge-m3'` y `'multilingual-e5-large'`

In [ ]:
# ── PARÁMETRO CONFIGURABLE ─────────────────────────────────────────────────────
MODELO = 'bge-m3'   # Opciones: 'bge-m3' | 'multilingual-e5-large'
METODO = 'ward'     # Método de linkage para clustering jerárquico
# ──────────────────────────────────────────────────────────────────────────────

# Cargar dataset maestro de carreras
df_carreras = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'carreras_homologas.csv')
df_carreras['perfil_egreso']      = df_carreras['perfil_egreso'].fillna('')
df_carreras['perfil_profesional'] = df_carreras['perfil_profesional'].fillna('')

# Cargar top-25 habilidades ESCO por carrera
topk_files = sorted(glob.glob(str(PROCESSED / f'esco_top25_{MODELO}_*.csv')))
if not topk_files:
    raise FileNotFoundError(f'No se encontró esco_top25_{MODELO}_*.csv — ejecuta primero esco_mapping.py')
df_topk = pd.read_csv(topk_files[-1])

# Cargar vectores de habilidades (matriz carrera × ESCO)
vec_files = sorted(glob.glob(str(PROCESSED / f'esco_vectors_{MODELO}_*.npy')))
if not vec_files:
    raise FileNotFoundError(f'No se encontró esco_vectors_{MODELO}_*.npy — ejecuta primero esco_mapping.py')
sim_matrix = np.load(vec_files[-1])  # shape: (163, 4241)

# Calcular similitud carrera vs carrera (163 × 163)
# Usamos cosine_similarity sobre los vectores ESCO de cada carrera
sim_cc = cosine_similarity(sim_matrix)
np.fill_diagonal(sim_cc, 1.0)

# Calcular matriz de distancias para clustering
dist_cc = np.clip(1 - sim_cc, 0, None)
np.fill_diagonal(dist_cc, 0)

print(f'✓ Datos cargados para modelo: {MODELO}')
print(f'  Carreras:             {len(df_carreras)}')
print(f'  Universidades:        {df_carreras["siglas"].nunique()}')
print(f'  Habilidades ESCO top: {df_topk["rank"].max()}')
print(f'  Vectores shape:       {sim_matrix.shape}  (carreras × habilidades ESCO)')
print(f'  Similitud shape:      {sim_cc.shape}  (carreras × carreras)')
print(f'  Rango similitud:      [{sim_cc[sim_cc < 1].min():.4f}, {sim_cc[sim_cc < 1].max():.4f}]')

---
## Sección 2 — Tabla de Habilidades ESCO por Carrera

**Qué hace:** Muestra las principales habilidades ESCO asociadas a cada carrera de la EPN, ordenadas por score de similitud semántica.

**Cómo interpretarlo:** Cada fila es una carrera EPN. Las columnas muestran las habilidades ESCO más cercanas semánticamente a los perfiles de egreso y profesional de esa carrera. Un score cercano a 1.0 significa que el texto de esa habilidad ESCO es muy similar semánticamente al perfil de la carrera.

**Por qué es útil:** Permite identificar qué competencias caracterizan a cada carrera de forma objetiva y estandarizada, usando el vocabulario común de ESCO. Esto hace que los resultados sean comparables entre universidades y países.

**Parámetro configurable:** `TOP_K` — número de habilidades a mostrar por carrera

In [ ]:
# ── PARÁMETRO CONFIGURABLE ─────────────────────────────────────────────────────
TOP_K       = 10     # Número de habilidades a mostrar por carrera
UNIVERSIDAD = 'EPN'  # Filtrar por universidad (ej: 'EPN', 'ESPOL', 'UPS', o None para todas)
# ──────────────────────────────────────────────────────────────────────────────

# Filtrar datos
df_filtrado = df_topk[df_topk['rank'] <= TOP_K].copy()
if UNIVERSIDAD:
    df_filtrado = df_filtrado[df_filtrado['siglas'] == UNIVERSIDAD]

# Tabla pivot: carreras × rank
pivot = df_filtrado.pivot_table(
    index='nombre', columns='rank',
    values='skill_label', aggfunc='first'
)
pivot.columns = [f'#{int(i)}' for i in pivot.columns]

# Mostrar tabla en notebook
print(f'Top-{TOP_K} habilidades ESCO — {UNIVERSIDAD if UNIVERSIDAD else "Todas las universidades"} | Modelo: {MODELO}')
print('=' * 80)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_columns', TOP_K + 1)
display(pivot)

# Guardar PNG
fig, ax = plt.subplots(figsize=(24, max(6, len(pivot) * 0.5 + 2)))
ax.axis('off')
fig.patch.set_facecolor('#fafafa')

tabla = ax.table(
    cellText=pivot.values,
    rowLabels=pivot.index,
    colLabels=pivot.columns,
    cellLoc='left', loc='center',
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(7.5)
tabla.auto_set_column_width(col=list(range(len(pivot.columns))))

for (row, col), cell in tabla.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#1e429f')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f0f4ff')
    else:
        cell.set_facecolor('#ffffff')
    cell.set_edgecolor('#e5e7eb')

ax.set_title(
    f'Top-{TOP_K} habilidades ESCO — {UNIVERSIDAD if UNIVERSIDAD else "Todas"} | Modelo: {MODELO}',
    fontsize=13, fontweight='bold', pad=16, color='#111827',
)

png_path = OUTPUT_DIR / f'tabla_habilidades_{MODELO}_{UNIVERSIDAD or "todas"}_{date.today().isoformat()}.png'
fig.savefig(png_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'\n✓ PNG guardado: {png_path}')

### 2.1 — Scores de similitud por carrera EPN

**Qué hace:** Muestra los scores numéricos de las habilidades más cercanas para una carrera específica.

**Cómo interpretarlo:** Un score de 0.85 significa que el embedding de esa habilidad ESCO es muy similar al embedding del perfil de la carrera. Scores por encima de 0.75 indican habilidades claramente presentes en el perfil. Scores entre 0.60-0.75 son habilidades relacionadas pero no centrales.

In [ ]:
# ── PARÁMETRO CONFIGURABLE ─────────────────────────────────────────────────────
CARRERA_VER = 'COMPUTACIÓN'   # Nombre exacto de la carrera a inspeccionar
SIGLAS_VER  = 'EPN'           # Universidad
# ──────────────────────────────────────────────────────────────────────────────

df_carrera = df_topk[
    (df_topk['nombre'] == CARRERA_VER) &
    (df_topk['siglas'] == SIGLAS_VER)
].sort_values('rank')

if df_carrera.empty:
    print(f'No se encontró {CARRERA_VER} — {SIGLAS_VER}')
else:
    print(f'Habilidades ESCO para: {CARRERA_VER} — {SIGLAS_VER} | Modelo: {MODELO}')
    print('=' * 60)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(
        df_carrera['skill_label'].str[:50],
        df_carrera['score'],
        color=get_color(SIGLAS_VER),
        alpha=0.85,
    )
    ax.set_xlabel('Score de similitud semántica', fontsize=11)
    ax.set_title(
        f'Top habilidades ESCO\n{CARRERA_VER} — {SIGLAS_VER} | {MODELO}',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlim(df_carrera['score'].min() * 0.95, df_carrera['score'].max() * 1.02)
    ax.invert_yaxis()
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    for bar, score in zip(bars, df_carrera['score']):
        ax.text(score + 0.001, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=8)
    
    plt.tight_layout()
    png_path = OUTPUT_DIR / f'skills_{CARRERA_VER}_{SIGLAS_VER}_{MODELO}_{date.today().isoformat()}.png'
    fig.savefig(png_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n✓ PNG guardado: {png_path}')

---
## Sección 3 — Heatmap de Similitud entre Carreras

**Qué hace:** Genera una matriz visual de similitud entre las 163 carreras. Cada celda muestra qué tan similares son dos carreras en términos de sus vectores de habilidades ESCO. Las filas y columnas están ordenadas por clustering jerárquico para que las carreras similares queden juntas.

**Cómo interpretarlo:**
- **Verde oscuro** → carreras muy similares (mismo nombre, misma área)
- **Amarillo** → similitud media (áreas relacionadas, ej: Computación y Sistemas)
- **Rojo** → carreras muy distintas (ej: Administración vs Ingeniería Civil)
- Los **bloques diagonales** verdes indican clusters de carreras similares
- El colormap usa **rango dinámico** (TwoSlopeNorm) para mostrar diferencias reales

**Por qué es útil:** Permite identificar de un vistazo qué carreras son curricular y competencialmente equivalentes entre universidades, y qué tan distintas son las diferentes áreas de conocimiento.

**Parámetro configurable:** `DPI` — resolución de la imagen (150 para pantalla, 300 para impresión)

In [ ]:
# ── PARÁMETRO CONFIGURABLE ─────────────────────────────────────────────────────
DPI = 150   # Resolución: 150 para pantalla, 300 para impresión en tesis
# ──────────────────────────────────────────────────────────────────────────────

# Clustering jerárquico para ordenar filas/columnas
Z = linkage(squareform(dist_cc, checks=False), method=METODO)
orden = leaves_list(Z)

# Reordenar matriz según clustering
sim_ordenada = sim_cc[np.ix_(orden, orden)]

# Etiquetas cortas ordenadas
etiquetas = [
    f"{df_carreras['nombre'].iloc[i][:28]}{'...' if len(df_carreras['nombre'].iloc[i]) > 28 else ''} — {df_carreras['siglas'].iloc[i]}"
    for i in orden
]

# Rango dinámico: excluir diagonal para calcular min/max real
mask    = ~np.eye(sim_ordenada.shape[0], dtype=bool)
vmin_r  = sim_ordenada[mask].min()
vmax_r  = sim_ordenada[mask].max()
vcenter = (vmin_r + vmax_r) / 2
norm    = TwoSlopeNorm(vmin=vmin_r, vcenter=vcenter, vmax=vmax_r)

print(f'Rango de similitud (sin diagonal): [{vmin_r:.4f}, {vmax_r:.4f}]')
print(f'Centro del colormap:               {vcenter:.4f}')

# Figura
n = len(df_carreras)
fig_size = max(20, n * 0.18)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
fig.patch.set_facecolor('#fafafa')

im = ax.imshow(sim_ordenada, cmap='RdYlGn', norm=norm,
               aspect='auto', interpolation='nearest')

plt.colorbar(im, ax=ax, label='Similitud coseno sobre vectores ESCO',
             shrink=0.6, pad=0.02)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(etiquetas, rotation=90, fontsize=5.5, ha='right')
ax.set_yticklabels(etiquetas, fontsize=5.5)

# Colorear etiquetas por universidad
for i, tick in enumerate(ax.get_yticklabels()):
    tick.set_color(get_color(df_carreras['siglas'].iloc[orden[i]]))
for i, tick in enumerate(ax.get_xticklabels()):
    tick.set_color(get_color(df_carreras['siglas'].iloc[orden[i]]))

ax.set_title(
    f'Matriz de similitud curricular — Nivel 3: Vectores ESCO\n'
    f'Modelo: {MODELO} · Ordenado por clustering Ward · '
    f'{n} carreras · {df_carreras["siglas"].nunique()} universidades',
    fontsize=12, fontweight='bold', color='#111827', pad=14,
)

plt.tight_layout(pad=1.5)

png_path = OUTPUT_DIR / f'heatmap_{MODELO}_{date.today().isoformat()}.png'
pdf_path = OUTPUT_DIR / f'heatmap_{MODELO}_{date.today().isoformat()}.pdf'
fig.savefig(png_path, dpi=DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
fig.savefig(pdf_path, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'\n✓ PNG guardado: {png_path}')
print(f'✓ PDF guardado: {pdf_path}')

---
## Sección 4 — Métricas de Evaluación

**Qué hace:** Calcula tres métricas cuantitativas para evaluar la calidad del clustering producido por cada modelo:

### Métricas calculadas

**1. Cophenetic Correlation**  
Mide qué tan bien el dendrograma preserva las distancias originales entre carreras. Es decir, si dos carreras estaban muy cerca en la matriz de distancias, ¿también se fusionan pronto en el dendrograma?  
- Rango: [0, 1] — más alto es mejor  
- > 0.8: Excelente | 0.7–0.8: Bueno | 0.6–0.7: Aceptable | < 0.6: Débil

**2. Silhouette Score**  
Mide cohesión y separación de los clusters. Para cada carrera, compara qué tan similar es a las carreras de su propio cluster versus las de otros clusters.  
- Rango: [-1, 1] — más alto es mejor  
- > 0.5: Clusters bien definidos | 0.25–0.5: Estructura débil | < 0.25: Sin estructura clara  
- Se evalúa para k=5, 7, 10 y 15 clusters

**3. Pureza de Clusters (k=10)**  
Para cada cluster, calcula qué porcentaje de carreras comparten el nombre más frecuente en ese cluster (ground truth = nombre de carrera).  
- Rango: [0, 1] — más alto es mejor  
- **Nota:** Esta métrica es estricta porque trata nombres ligeramente distintos como categorías diferentes (ej: 'Computación' ≠ 'Ingeniería en Ciencias de la Computación'). Subestima la pureza real.

**Cuándo ejecutar:** Después de cargar los datos en la Sección 1. Cambia `MODELO` en la Sección 1 para comparar métricas entre modelos.

In [ ]:
print(f'Calculando métricas para modelo: {MODELO}')
print('=' * 50)

dist_condensed = squareform(dist_cc, checks=False)
Z_met = linkage(dist_condensed, method=METODO)

# 1. Cophenetic correlation
cophenetic_corr, _ = cophenet(Z_met, dist_condensed)
interp_coph = (
    'Excelente (>0.8)' if cophenetic_corr > 0.8 else
    'Bueno (0.7-0.8)'  if cophenetic_corr > 0.7 else
    'Aceptable (0.6-0.7)' if cophenetic_corr > 0.6 else
    'Débil (<0.6)'
)
print(f'\n1. Cophenetic Correlation: {cophenetic_corr:.4f}  →  {interp_coph}')

# 2. Silhouette scores
print('\n2. Silhouette Scores:')
sil_scores = {}
for k in [5, 7, 10, 15]:
    labels = fcluster(Z_met, k, criterion='maxclust')
    score  = silhouette_score(dist_cc, labels, metric='precomputed')
    sil_scores[k] = score
    interp = (
        'Bien definidos' if score > 0.5 else
        'Estructura débil' if score > 0.25 else
        'Sin estructura clara'
    )
    print(f'   k={k:2d}: {score:+.4f}  →  {interp}')

# 3. Pureza de clusters (k=10)
labels_10 = fcluster(Z_met, 10, criterion='maxclust')
df_eval   = df_carreras[['siglas','nombre']].copy()
df_eval['cluster']    = labels_10
df_eval['nombre_norm'] = df_eval['nombre'].str.lower().str.strip()

pureza_total = 0
n_clusters   = df_eval['cluster'].nunique()
for cid in df_eval['cluster'].unique():
    grupo = df_eval[df_eval['cluster'] == cid]
    pureza_total += grupo['nombre_norm'].value_counts().iloc[0] / len(grupo)
pureza = pureza_total / n_clusters

interp_pur = (
    'Alta (>0.8)'     if pureza > 0.8 else
    'Media (0.6-0.8)' if pureza > 0.6 else
    'Baja (<0.6)'
)
print(f'\n3. Pureza de Clusters (k=10): {pureza:.4f}  →  {interp_pur}')
print('   Nota: métrica estricta — nombres distintos cuentan como categorías distintas')

# Guardar resultados
metricas = {
    'modelo': MODELO, 'metodo': METODO,
    'n_carreras': len(df_carreras),
    'cophenetic': round(cophenetic_corr, 4),
    'silhouette': {f'k={k}': round(v, 4) for k, v in sil_scores.items()},
    'pureza_k10': round(pureza, 4),
}
json_path = OUTPUT_DIR / f'metricas_{MODELO}_{date.today().isoformat()}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)
print(f'\n✓ Métricas guardadas: {json_path}')

---
## Sección 5 — Comparación de Modelos

**Qué hace:** Compara las métricas de todos los modelos disponibles en `data/processed/` en una sola tabla.

**Cómo interpretarlo:** Permite identificar qué modelo produce clusters más coherentes para el dataset de carreras universitarias ecuatorianas. Un modelo es mejor si tiene mayor cophenetic correlation, mayor Silhouette score y mayor pureza.

**Cuándo ejecutar:** Después de haber ejecutado `esco_mapping.py` y las métricas para ambos modelos (BGE-M3 y E5-Large).

**Nota importante:** El Silhouette score usa la matriz de distancias precalculada (`metric='precomputed'`) para que la comparación sea justa entre modelos.

In [ ]:
# Detectar todos los modelos disponibles
modelos_disponibles = []
for f in sorted(glob.glob(str(PROCESSED / 'esco_vectors_*_*.npy'))):
    nombre = Path(f).stem.replace('esco_vectors_','').rsplit('_',1)[0]
    if nombre not in modelos_disponibles:
        modelos_disponibles.append(nombre)

print(f'Modelos disponibles: {modelos_disponibles}')
print('Calculando métricas para todos...')

resultados = []
for modelo in modelos_disponibles:
    # Cargar vectores
    vec_f = sorted(glob.glob(str(PROCESSED / f'esco_vectors_{modelo}_*.npy')))[-1]
    sm    = np.load(vec_f)
    sc    = cosine_similarity(sm)
    np.fill_diagonal(sc, 1.0)
    dc    = np.clip(1 - sc, 0, None)
    np.fill_diagonal(dc, 0)

    # Métricas
    dc_cond = squareform(dc, checks=False)
    Zm      = linkage(dc_cond, method=METODO)
    coph, _ = cophenet(Zm, dc_cond)

    sil = {}
    for k in [5, 7, 10, 15]:
        lbl    = fcluster(Zm, k, criterion='maxclust')
        sil[k] = round(silhouette_score(dc, lbl, metric='precomputed'), 4)

    lbl10 = fcluster(Zm, 10, criterion='maxclust')
    df_e  = df_carreras[['nombre']].copy()
    df_e['cluster'] = lbl10
    df_e['n_norm']  = df_e['nombre'].str.lower().str.strip()
    pt = sum(
        df_e[df_e['cluster']==c]['n_norm'].value_counts().iloc[0] / len(df_e[df_e['cluster']==c])
        for c in df_e['cluster'].unique()
    ) / df_e['cluster'].nunique()

    resultados.append({
        'Modelo':        modelo,
        'Cophenetic':    round(coph, 4),
        'Silhouette k=5': sil[5],
        'Silhouette k=7': sil[7],
        'Silhouette k=10':sil[10],
        'Silhouette k=15':sil[15],
        'Pureza k=10':   round(pt, 4),
    })
    print(f'  ✓ {modelo}: Cophenetic={coph:.4f} | Silhouette k=10={sil[10]:.4f} | Pureza={pt:.4f}')

df_comp = pd.DataFrame(resultados)
print('\n=== TABLA COMPARATIVA ===')
display(df_comp.style
    .background_gradient(subset=['Cophenetic','Silhouette k=10','Pureza k=10'], cmap='RdYlGn')
    .format(precision=4)
)

# Guardar
json_path = OUTPUT_DIR / f'comparacion_modelos_{date.today().isoformat()}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print(f'\n✓ Comparación guardada: {json_path}')

---
## Sección 6 — Interpretación y Conclusiones

**Qué hace:** Genera un resumen automático de los hallazgos basado en las métricas calculadas, listo para incluir como base del análisis en la tesis.

**Cómo usarlo:** Ejecuta esta celda después de la Sección 5. El texto generado es un punto de partida — edítalo y amplíalo con tu propia interpretación y conocimiento del dominio.

In [ ]:
if not resultados:
    print('Ejecuta primero la Sección 5 para generar la comparación de modelos.')
else:
    # Identificar mejor modelo por cada métrica
    df_c = pd.DataFrame(resultados)
    mejor_coph = df_c.loc[df_c['Cophenetic'].idxmax(), 'Modelo']
    mejor_sil  = df_c.loc[df_c['Silhouette k=10'].idxmax(), 'Modelo']
    mejor_pur  = df_c.loc[df_c['Pureza k=10'].idxmax(), 'Modelo']

    print('=' * 70)
    print('RESUMEN DE HALLAZGOS — Nivel 3: Vectores ESCO')
    print('=' * 70)
    print(f"""
Se evaluaron {len(modelos_disponibles)} modelos de embeddings multilingües para representar
{len(df_carreras)} carreras de {df_carreras['siglas'].nunique()} universidades ecuatorianas
mediante vectores de habilidades ESCO (taxonomía transversal y cross-sector,
{sim_matrix.shape[1]} habilidades).

RESULTADOS POR MÉTRICA:
  • Cophenetic correlation:  mejor modelo = {mejor_coph}
  • Silhouette score (k=10): mejor modelo = {mejor_sil}
  • Pureza de clusters:      mejor modelo = {mejor_pur}

OBSERVACIONES:
  • Los dendrogramas muestran agrupamientos coherentes con las áreas de
    conocimiento esperadas (ingeniería civil, computación, administración,
    química, telecomunicaciones).

  • La pureza de clusters es una métrica estricta que subestima la coherencia
    real porque trata nombres ligeramente distintos para carreras equivalentes
    como categorías diferentes (ej: 'Computación' ≠ 'Ingeniería en Ciencias
    de la Computación').

  • Comparado con el baseline TF-IDF (Nivel 1), los vectores ESCO producen
    agrupamientos más coherentes curricularmente porque capturan similitud
    semántica a nivel de competencias, no solo coincidencia léxica.

LIMITACIONES:
  • El análisis usa solo habilidades transversal y cross-sector de ESCO.
    Incluir habilidades sector-specific podría mejorar la discriminación
    entre carreras de ingeniería.

  • El dataset actual cubre solo Ecuador (163 carreras). Los resultados
    pueden variar al incorporar carreras de LATAM y EEUU.

PRÓXIMOS PASOS:
  • Ampliar dataset con Grupos 2 y 3 de Ecuador (~195 carreras adicionales)
  • Incorporar carreras de 8 países OCDE (LATAM + EEUU)
  • Avanzar al Nivel 4: modelo de embeddings propio para habilidades
  • Avanzar al Nivel 5: comparación con ofertas laborales
""")
    print('=' * 70)